In [6]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Load dataset
data = pd.read_csv("dataset/network.csv", header=None)

# Label column is second-to-last
label_column = data.columns[-2]

print("Original shape:", data.shape)

# Drop nulls
data = data.dropna()

print("After dropping nulls:", data.shape)

# Encode labels: normal = 0, everything else = 1
data[label_column] = data[label_column].apply(lambda x: 0 if x == "normal" else 1)

# Features and target
X = data.drop(label_column, axis=1)
y = data[label_column]

# One-hot encode categorical columns
X = pd.get_dummies(X)

# Make sure all column names are strings (fixes sklearn error)
X.columns = X.columns.astype(str)

print("Feature matrix shape after encoding:", X.shape)
print("Target shape:", y.shape)

Original shape: (125973, 43)
After dropping nulls: (125973, 43)
Feature matrix shape after encoding: (125973, 123)
Target shape: (125973,)


In [7]:
# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Normalization complete!")
print("Scaled feature shape:", X_scaled.shape)

Normalization complete!
Scaled feature shape: (125973, 123)


In [8]:
from sklearn.model_selection import train_test_split

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (100778, 123)
Test shape: (25195, 123)


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Predict
rf_pred = rf_model.predict(X_test)

# Evaluate
rf_acc = accuracy_score(y_test, rf_pred)

print("=== RANDOM FOREST RESULTS ===")
print("Accuracy:", rf_acc)
print(classification_report(y_test, rf_pred))

=== RANDOM FOREST RESULTS ===
Accuracy: 0.9995634054375868
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     13422
           1       1.00      1.00      1.00     11773

    accuracy                           1.00     25195
   macro avg       1.00      1.00      1.00     25195
weighted avg       1.00      1.00      1.00     25195



In [11]:
from sklearn.ensemble import IsolationForest

# Use only benign traffic (label = 0) from training data
X_train_benign = X_train[y_train == 0]

# Train Isolation Forest
iso_model = IsolationForest(
    contamination=0.1,
    random_state=42
)

iso_model.fit(X_train_benign)

print("Isolation Forest trained on benign-only data!")

Isolation Forest trained on benign-only data!


In [12]:
from sklearn.ensemble import IsolationForest

# Use only benign traffic (label = 0) from training data
X_train_benign = X_train[y_train == 0]

# Train Isolation Forest
iso_model = IsolationForest(
    contamination=0.1,
    random_state=42
)

iso_model.fit(X_train_benign)

print("Isolation Forest trained on benign-only data!")

Isolation Forest trained on benign-only data!


In [16]:
# Predict anomalies on test data
iso_pred_raw = iso_model.predict(X_test)

# Convert: 1 = normal -> 0,  -1 = anomaly -> 1
iso_pred = [0 if p == 1 else 1 for p in iso_pred_raw]

# Accuracy
from sklearn.metrics import accuracy_score, classification_report

iso_acc = accuracy_score(y_test, iso_pred)

print("=== ISOLATION FOREST RESULTS ===")
print("Accuracy:", iso_acc)
print(classification_report(y_test, iso_pred))

=== ISOLATION FOREST RESULTS ===
Accuracy: 0.9179202222663226
              precision    recall  f1-score   support

           0       0.95      0.90      0.92     13422
           1       0.89      0.94      0.91     11773

    accuracy                           0.92     25195
   macro avg       0.92      0.92      0.92     25195
weighted avg       0.92      0.92      0.92     25195



In [17]:
# Predict anomalies on test data
iso_pred_raw = iso_model.predict(X_test)

# Convert: 1 = normal -> 0,  -1 = anomaly -> 1
iso_pred = [0 if p == 1 else 1 for p in iso_pred_raw]

# Accuracy
from sklearn.metrics import accuracy_score, classification_report

iso_acc = accuracy_score(y_test, iso_pred)

print("=== ISOLATION FOREST RESULTS ===")
print("Accuracy:", iso_acc)
print(classification_report(y_test, iso_pred))

=== ISOLATION FOREST RESULTS ===
Accuracy: 0.9179202222663226
              precision    recall  f1-score   support

           0       0.95      0.90      0.92     13422
           1       0.89      0.94      0.91     11773

    accuracy                           0.92     25195
   macro avg       0.92      0.92      0.92     25195
weighted avg       0.92      0.92      0.92     25195

